# Aggregate Results


## 1. Load Raw Experiment Results

Loads the per-run results produced by `run_experiment.ipynb`.


In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.insert(0, "..")
import config

df = pd.read_csv(config.AGGREGATED_RESULTS_DIR / "results.csv")
print(f"Loaded {len(df)} rows")
df.head()


## 2. Aggregate Results By Task

Calculates mean and standard deviation for quality, execution time, tokens, and cost for each scenario, task variant, and architecture.


In [ ]:
agg = df.groupby(["scenario", "variant", "architecture"]).agg(
    quality_mean=("quality_score", "mean"),
    quality_std=("quality_score", "std"),
    time_mean=("time_seconds", "mean"),
    time_std=("time_seconds", "std"),
    tokens_mean=("tokens", "mean"),
    tokens_std=("tokens", "std"),
    cost_mean=("cost", "mean"),
    cost_std=("cost", "std"),
    n_runs=("run", "count"),
).round(4).reset_index()

out_path = config.AGGREGATED_RESULTS_DIR / "results_aggregated.csv"
agg.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
agg


## 3. Aggregate Results By Scenario

Creates the scenario-level summary table used in the experiment analysis.


In [ ]:
scenario_names = {
    "context_pollution": "Konteksta piesarnojums",
    "parallelization": "Paralela izpilde",
    "verification": "Verifikacija",
}

architecture_names = {
    "single_agent": "SAS",
    "multi_agent": "MAS",
}

scenario_agg = df.groupby(["scenario", "architecture"]).agg(
    quality_mean=("quality_score", "mean"),
    time_mean=("time_seconds", "mean"),
    tokens_mean=("tokens", "mean"),
    cost_mean=("cost", "mean"),
).round(4).reset_index()

scenario_agg["Scenarijs"] = scenario_agg["scenario"].map(scenario_names)
scenario_agg["Arhitektura"] = scenario_agg["architecture"].map(architecture_names)

scenario_agg = scenario_agg[[
    "Scenarijs",
    "Arhitektura",
    "quality_mean",
    "time_mean",
    "tokens_mean",
    "cost_mean",
]]

scenario_agg.columns = [
    "Scenarijs",
    "Arhitektura",
    "Quality mean",
    "Time mean (s)",
    "Tokens mean",
    "Cost mean (USD)",
]

out_path = config.AGGREGATED_RESULTS_DIR / "results_aggregated_by_scenario.csv"
scenario_agg.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
scenario_agg


## 4. Calculate Overall Score By Task

Calculates the quality-gated overall score for each task and architecture. Stability is based on the average coefficient of variation for quality, execution time, and cost. The formula is `Overall = Quality * (0.50 + 0.20 * Stability + 0.15 * TimeEfficiency + 0.15 * CostEfficiency)`.


In [ ]:
WEIGHTS = {
    "stability": 0.20,
    "time_efficiency": 0.15,
    "cost_efficiency": 0.15,
}

score = agg.copy()
score["quality_std"] = score["quality_std"].fillna(0.0)
score["time_std"] = score["time_std"].fillna(0.0)
score["cost_std"] = score["cost_std"].fillna(0.0)

score["Quality"] = score["quality_mean"]

score["CVQuality"] = 1.0
quality_mask = score["quality_mean"] > 0
score.loc[quality_mask, "CVQuality"] = score.loc[quality_mask, "quality_std"] / score.loc[quality_mask, "quality_mean"]
score["CVTime"] = score["time_std"] / score["time_mean"]
score["CVCost"] = score["cost_std"] / score["cost_mean"]

score["Stability"] = (
    1.0 - score[["CVQuality", "CVTime", "CVCost"]].mean(axis=1)
).clip(lower=0.0, upper=1.0)

score["min_time_in_task"] = score.groupby(["scenario", "variant"])["time_mean"].transform("min")
score["min_cost_in_task"] = score.groupby(["scenario", "variant"])["cost_mean"].transform("min")
score["TimeEfficiency"] = score["min_time_in_task"] / score["time_mean"]
score["CostEfficiency"] = score["min_cost_in_task"] / score["cost_mean"]

score["Overall"] = score["Quality"] * (
    0.50
    + WEIGHTS["stability"] * score["Stability"]
    + WEIGHTS["time_efficiency"] * score["TimeEfficiency"]
    + WEIGHTS["cost_efficiency"] * score["CostEfficiency"]
)

score_by_task = score[[
    "scenario",
    "variant",
    "architecture",
    "Quality",
    "Stability",
    "TimeEfficiency",
    "CostEfficiency",
    "Overall",
]].copy()

score_by_task.columns = [
    "Scenario",
    "Variant",
    "Architecture",
    "Quality",
    "Stability",
    "TimeEfficiency",
    "CostEfficiency",
    "Overall",
]

score_by_task = score_by_task.round(4)

out_path = config.AGGREGATED_RESULTS_DIR / "overall_scores_by_task.csv"
score_by_task.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
score_by_task


## 5. Summarize Overall Score By Scenario

Averages task-level overall scores by scenario and architecture, then identifies the winning architecture.


In [ ]:
scenario_scores = score_by_task.groupby(["Scenario", "Architecture"], as_index=False)["Overall"].mean()
scenario_scores["Architecture"] = scenario_scores["Architecture"].map(architecture_names)

summary = scenario_scores.pivot(index="Scenario", columns="Architecture", values="Overall").reset_index()
summary = summary[["Scenario", "MAS", "SAS"]]

def select_winner(row):
    if row["MAS"] > row["SAS"]:
        return "MAS"
    if row["SAS"] > row["MAS"]:
        return "SAS"
    return "Tie"

summary["Winner"] = summary.apply(select_winner, axis=1)

overall_row = {
    "Scenario": "Overall",
    "MAS": summary["MAS"].mean(),
    "SAS": summary["SAS"].mean(),
}
overall_row["Winner"] = "MAS" if overall_row["MAS"] > overall_row["SAS"] else "SAS" if overall_row["SAS"] > overall_row["MAS"] else "Tie"
summary = pd.concat([summary, pd.DataFrame([overall_row])], ignore_index=True)
summary = summary.round({"MAS": 4, "SAS": 4})

out_path = config.AGGREGATED_RESULTS_DIR / "overall_scores_by_scenario.csv"
summary.to_csv(out_path, index=False)
print(f"Saved: {out_path}")
summary
